<a href="https://colab.research.google.com/github/Godstouch/GNN-Student-Risk-Prediction-/blob/main/TrackB_Q23_synthetic_peer_effect_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch_geometric

In [ ]:
"""
Track B — Synthetic peer-effect validation (closes Q23).
No real pupil data is used anywhere in this file.

Design:
  - SBM graph = 3 synthetic "schools" sized from the study sample's school counts
    (556 / 269 / 175); only these counts come from real data
  - label depends on own baseline + neighbours' baselines (peer_weight)
  - X contains only a NOISY copy of own baseline + distractor noise columns
    (drivers `baseline` and `neighbour_term` are kept OUT of X)
  - peer_weight = 0 is the null control: graph models should NOT beat the MLP there
"""
import numpy as np
import networkx as nx
import torch
from torch_geometric.data import Data
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

PEER_WEIGHTS = [0.0, 0.2, 0.4, 0.6]
SEEDS = list(range(10))            # confirm these match the seeds used elsewhere
MODELS = ["gated_gat", "gat_no_gate_ablation", "graphsage", "mlp", "logreg"]


# Block sizes = real school sizes (summary counts only; no pupil rows used)
SCHOOL_SIZES = (556, 269, 175)   # Weweso MA, Ayeduase RC, Shining Star Preparatory


def make_synthetic(seed, peer_weight=0.4, sizes=SCHOOL_SIZES,
                   mean_deg_in=15, p_out=0.0045, noise_sd=0.5,
                   n_distractors=26, positive_rate=0.15):
    rng = np.random.default_rng(seed)

    # Step 1 — synthetic network, one block per real school.
    # Within-school probability is set per block so every school has the same
    # expected within-school degree (a fixed p would make big schools far denser).
    k = len(sizes)
    probs = [[mean_deg_in / (sizes[i] - 1) if i == j else p_out for j in range(k)]
             for i in range(k)]
    G = nx.stochastic_block_model(list(sizes), probs, seed=seed)
    n = G.number_of_nodes()

    # Step 2 — inject a known peer effect
    baseline = rng.normal(0, 1, size=n)
    neighbour_term = np.array([
        baseline[list(G.neighbors(i))].mean() if G.degree(i) > 0 else 0.0
        for i in range(n)
    ])
    # Standardise: averaging over ~20 neighbours shrinks variance ~20x, which
    # would make the peer effect negligible regardless of peer_weight.
    neighbour_term = (neighbour_term - neighbour_term.mean()) / neighbour_term.std()
    true_risk = (1 - peer_weight) * baseline + peer_weight * neighbour_term
    y = (true_risk >= np.quantile(true_risk, 1 - positive_rate)).astype(int)

    # Step 3 — features: noisy own-signal + distractors, NO driver columns
    own_noisy = baseline + rng.normal(0, noise_sd, size=n)
    distractors = rng.normal(0, 1, size=(n, n_distractors))
    X = np.column_stack([own_noisy, distractors]).astype(np.float32)

    # Undirected edges -> both directions for PyG
    e = np.array(G.edges()).T
    edge_index = torch.tensor(np.concatenate([e, e[::-1]], axis=1),
                              dtype=torch.long)

    # 60/20/20 node split
    perm = rng.permutation(n)
    n_tr, n_va = int(0.6 * n), int(0.2 * n)
    masks = {k: torch.zeros(n, dtype=torch.bool) for k in ("train", "val", "test")}
    masks["train"][perm[:n_tr]] = True
    masks["val"][perm[n_tr:n_tr + n_va]] = True
    masks["test"][perm[n_tr + n_va:]] = True

    data = Data(x=torch.tensor(X), edge_index=edge_index,
                y=torch.tensor(y, dtype=torch.long),
                train_mask=masks["train"], val_mask=masks["val"],
                test_mask=masks["test"])
    drivers = {"baseline": baseline, "neighbour_term": neighbour_term}  # never fed to models
    return data, drivers


def oracle_and_floor(data, drivers):
    """Ceiling: logreg on the true drivers. Floor: logreg on X with no graph."""
    tr, te = data.train_mask.numpy(), data.test_mask.numpy()
    y = data.y.numpy()
    D = np.column_stack([drivers["baseline"], drivers["neighbour_term"]])
    X = data.x.numpy()
    out = {}
    for name, feats in (("oracle_ceiling", D), ("no_graph_floor", X)):
        clf = LogisticRegression(max_iter=1000)
        clf.fit(feats[tr], y[tr])
        out[name] = f1_score(y[te], clf.predict(feats[te]), average="macro")
    return out


def sanity_checks(data, drivers):
    x0 = data.x[:, 0].numpy()
    r_own = np.corrcoef(x0, drivers["baseline"])[0, 1]
    r_nb = np.corrcoef(x0, drivers["neighbour_term"])[0, 1]
    print(f"  nodes: {data.num_nodes} | features: {data.x.shape[1]} | "
          f"mean degree: {data.edge_index.shape[1] / data.num_nodes:.1f}")
    print(f"  positive rate: {data.y.float().mean():.3f}")
    print(f"  corr(X_own, baseline)       = {r_own:.2f}  (should be <1: noisy, not leaked)")
    print(f"  corr(X_own, neighbour_term) = {r_nb:.2f}  (should be low: peer signal not in X)")
    assert r_own < 0.99, "Driver leaked into X"


def export_datasets(out_dir="trackB_synthetic_data", weights=PEER_WEIGHTS, seeds=SEEDS):
    """Save every dataset in the grid.
    <tag>.pt              PyG Data object for training (torch.load(..., weights_only=False))
    <tag>_nodes.csv       features, label, split per node (for inspection)
    <tag>_edges.csv       source/target pairs
    <tag>_DRIVERS_...csv  true baseline + neighbour_term — audit only, NEVER features
    """
    import os
    import pandas as pd
    os.makedirs(out_dir, exist_ok=True)
    for w in weights:
        for s in seeds:
            data, drivers = make_synthetic(seed=s, peer_weight=w)
            tag = f"w{w:.1f}_seed{s}"
            torch.save(data, f"{out_dir}/{tag}.pt")

            cols = ["x_own_noisy"] + [f"noise_{i + 1}" for i in range(data.x.shape[1] - 1)]
            nodes = pd.DataFrame(data.x.numpy(), columns=cols)
            nodes["label"] = data.y.numpy()
            nodes["split"] = np.select([data.train_mask.numpy(), data.val_mask.numpy()],
                                       ["train", "val"], "test")
            nodes.to_csv(f"{out_dir}/{tag}_nodes.csv", index_label="node_id")

            ei = data.edge_index.numpy()
            pd.DataFrame({"source": ei[0], "target": ei[1]}).to_csv(
                f"{out_dir}/{tag}_edges.csv", index=False)
            pd.DataFrame(drivers).to_csv(
                f"{out_dir}/{tag}_DRIVERS_do_not_use_as_features.csv", index_label="node_id")
    print(f"Saved {len(weights) * len(seeds)} datasets to {out_dir}/")
    return out_dir


if __name__ == "__main__":
    # Quick look before the full grid — share this output with Rabbi for design review
    for w in PEER_WEIGHTS:
        data, drivers = make_synthetic(seed=0, peer_weight=w)
        print(f"peer_weight={w}")
        sanity_checks(data, drivers)
        runs = [oracle_and_floor(*make_synthetic(seed=s, peer_weight=w))
                for s in range(5)]
        for key in ("oracle_ceiling", "no_graph_floor"):
            vals = [r[key] for r in runs]
            print(f"  {key}: {np.mean(vals):.3f} ± {np.std(vals):.3f}  (5 seeds)")

    # Full grid (Step 4) — plug in the team's existing training function
    # results = []
    # for w in PEER_WEIGHTS:
    #     for seed in SEEDS:
    #         data, _ = make_synthetic(seed=seed, peer_weight=w)
    #         for model in MODELS:
    #             m = train_and_evaluate(model=model, seed=seed, data=data)
    #             results.append({"peer_weight": w, "seed": seed, "model": model, **m})

peer_weight=0.0
  nodes: 1000 | features: 27 | mean degree: 17.8
  positive rate: 0.150
  corr(X_own, baseline)       = 0.89  (should be <1: noisy, not leaked)
  corr(X_own, neighbour_term) = 0.02  (should be low: peer signal not in X)
  oracle_ceiling: 0.980 ± 0.010  (5 seeds)
  no_graph_floor: 0.818 ± 0.013  (5 seeds)
peer_weight=0.2
  nodes: 1000 | features: 27 | mean degree: 17.8
  positive rate: 0.150
  corr(X_own, baseline)       = 0.89  (should be <1: noisy, not leaked)
  corr(X_own, neighbour_term) = 0.02  (should be low: peer signal not in X)
  oracle_ceiling: 0.980 ± 0.007  (5 seeds)
  no_graph_floor: 0.780 ± 0.040  (5 seeds)
peer_weight=0.4
  nodes: 1000 | features: 27 | mean degree: 17.8
  positive rate: 0.150
  corr(X_own, baseline)       = 0.89  (should be <1: noisy, not leaked)
  corr(X_own, neighbour_term) = 0.02  (should be low: peer signal not in X)
  oracle_ceiling: 0.982 ± 0.012  (5 seeds)
  no_graph_floor: 0.692 ± 0.022  (5 seeds)
peer_weight=0.6
  nodes: 1000 | fe

In [ ]:
import shutil
from google.colab import files
out = export_datasets()
shutil.make_archive(out, "zip", out)
files.download(out + ".zip")

Saved 40 datasets to trackB_synthetic_data/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>